# Lesson 9: Interactive Dashboards with Gradio 🎵

**Welcome back!** Today you'll learn to:
- Turn your Python code into **interactive web apps** anyone can use
- Use **Gradio** — a library that builds interfaces from Python functions
- Connect **pandas filtering and plotting** to live controls (dropdowns, sliders, checkboxes)
- Build a **music streaming dashboard** that lets users explore data without writing code

---

## 🔄 Quick Recap

In Lessons 7 and 8 you learned to:
- Load CSV data into a DataFrame with `pd.read_csv()`
- Filter rows: `df[df["column"] == "value"]`
- Group and summarise: `df.groupby("column")["value"].mean()`
- Create computed columns: `df["new"] = df["old"] * 4`
- Plot charts directly from DataFrames

Today you'll use **all of these skills** — but instead of running cells one by one, you'll wrap them in functions that power a live, interactive app.

**💬 Teams chat question:** If you could let someone explore a dataset by clicking buttons instead of writing code, what kind of data would you want to share?

---

## 🤔 Why Gradio?

So far, your Python programs only work for people who can read and run code. But what if you wanted to share your data analysis with:
- A friend who doesn't know Python?
- A teacher who just wants to explore the results?
- Anyone with a web browser?

**Gradio** turns a normal Python function into a web app with buttons, dropdowns, and sliders — automatically. You write the logic, Gradio builds the interface.

![Why Gradio — Before and After](https://raw.githubusercontent.com/simunaqv2/python-jupyter-course/main/lessons/lesson09/gradio_before_after.png)

Here's the pattern you'll use all lesson:

1. **Write a function** that takes input and returns output
2. **Wrap it** in a Gradio interface
3. **Launch it** — and you have a web app!

---

## 🎵 Today's Dataset: Music Streaming Survey

We've surveyed 30 students about their music listening habits. Let's load it and take a look.

First, let's install Gradio and import everything we need:

```python
!pip install gradio -q

import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

print("✅ All libraries loaded!")
```

Now let's load the music streaming dataset:

```python
url = "https://raw.githubusercontent.com/simunaqv2/python-jupyter-course/main/lessons/lesson09/music_streaming_survey.csv"
music = pd.read_csv(url)
music.head()
```

Let's get a quick overview:

```python
print(f"Rows: {music.shape[0]}, Columns: {music.shape[1]}")
print(f"\nColumns: {music.columns.tolist()}")
print(f"\nGenres: {music['genre'].unique().tolist()}")
print(f"Platforms: {music['platform'].unique().tolist()}")
print(f"Moods: {music['mood'].unique().tolist()}")
```

We have 30 students, 9 columns, and several categories to explore. This is the data we'll make interactive today!

---

## Part 1: Hello Gradio 👋

Let's start with the simplest possible Gradio app — a function that takes a name and returns a greeting.

### The Three-Step Pattern

Every Gradio app follows the same pattern:

![The Gradio Pattern — Function, Interface, Launch](https://raw.githubusercontent.com/simunaqv2/python-jupyter-course/main/lessons/lesson09/gradio_pattern.png)

| Step | What you do | Code |
|------|------------|------|
| 1. Write a function | Your Python logic | `def greet(name): ...` |
| 2. Create an interface | Tell Gradio what inputs/outputs to show | `gr.Interface(fn=..., inputs=..., outputs=...)` |
| 3. Launch it | Start the web app | `demo.launch()` |

```python
# Step 1: Write a function
def greet(name):
    return f"Hello, {name}! Welcome to the Music Dashboard 🎵"

# Step 2: Create the interface
demo = gr.Interface(
    fn=greet,
    inputs=gr.Textbox(label="Enter your name"),
    outputs=gr.Textbox(label="Greeting")
)

# Step 3: Launch!
demo.launch()
```

🎉 **You just built a web app!** Type your name in the box, click Submit, and see the result.

Notice how Gradio automatically created:
- A text input box (because we said `inputs=gr.Textbox(...)`)
- A text output box (because we said `outputs=gr.Textbox(...)`)
- A Submit button
- A Clear button

All you wrote was a normal Python function — Gradio built the rest!

---

## Part 2: Gradio with Numbers — Listener Categoriser 🎧

Let's make something more useful. Remember the `categorise_gamer` function from Lesson 6? Let's build a similar one for music listening and connect it to a **slider**.

```python
def categorise_listener(hours_per_week):
    monthly = hours_per_week * 4
    
    if hours_per_week >= 18:
        category = "🎶 Super Listener"
    elif hours_per_week >= 10:
        category = "🎵 Regular Listener"
    else:
        category = "🔈 Casual Listener"
    
    return f"{category}\n\nYou listen about {hours_per_week} hours/week ({monthly} hours/month)"

demo = gr.Interface(
    fn=categorise_listener,
    inputs=gr.Slider(minimum=0, maximum=30, step=1, value=10, label="Hours of music per week"),
    outputs=gr.Textbox(label="Your listening category")
)

demo.launch()
```

Try dragging the slider! Notice how:
- The slider sends a **number** to your function (not a string — no `int()` conversion needed!)
- Your function uses the same `if/elif/else` logic from Lesson 3
- The `return` value (from Lesson 6) appears in the output box

**Key insight:** Gradio handles the user input for you. You just write the function logic.

---

## Part 3: Connecting pandas to Gradio 🐼 + 🖥️

This is the big one. We're going to connect our music dataset to Gradio so users can **filter and visualise data interactively**.

### Returning Charts from Functions

Before we build the interface, there's one important trick: when Gradio needs to display a chart, your function must **return the figure object**, not call `plt.show()`.

Here's the pattern:

```python
# ❌ The old way (won't work with Gradio)
plt.bar(names, values)
plt.show()

# ✅ The Gradio way (return the figure)
fig, ax = plt.subplots()
ax.bar(names, values)
return fig
```

Think of it like this: `plt.show()` **displays** the chart on your screen. But Gradio needs you to **hand over** the chart so *it* can display it in the web app. That's what `return fig` does.

Let's try it with a function that filters music data by genre:

```python
def plot_genre(genre):
    # Filter the data (just like Lesson 7!)
    filtered = music[music["genre"] == genre]
    
    # Create the chart
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(filtered["student"], filtered["hours_per_week"], color="steelblue")
    ax.set_title(f"Listening Hours — {genre} Fans")
    ax.set_xlabel("Student")
    ax.set_ylabel("Hours per Week")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    
    return fig

# Get the list of genres from the data
genres = music["genre"].unique().tolist()

demo = gr.Interface(
    fn=plot_genre,
    inputs=gr.Dropdown(choices=genres, value=genres[0], label="Choose a genre"),
    outputs=gr.Plot(label="Listening Hours")
)

demo.launch()
```

🎉 **That's a live data explorer!** Try switching genres in the dropdown — the chart updates instantly.

Let's break down what happened:

| Part | What it does | Skills from |
|------|-------------|-------------|
| `music[music["genre"] == genre]` | Filters the DataFrame | Lesson 7 |
| `ax.bar(...)` | Creates a bar chart | Lesson 4 |
| `def plot_genre(genre):` | Wraps it in a function | Lesson 6 |
| `return fig` | Hands the chart to Gradio | **New today** |
| `gr.Dropdown(...)` | Creates a dropdown from our genre list | **New today** |

Almost everything here is revision — the only new pieces are `fig, ax` and the Gradio wrapper!

---

### 💪 Practice: Platform Explorer

Write a function called `plot_platform` that takes a platform name (Spotify, Apple Music, or YouTube Music) and returns a bar chart of listening hours for students on that platform.

**Your function should:**
1. Filter `music` by the chosen platform
2. Create a bar chart of `student` vs `hours_per_week`
3. Return the figure

Then connect it to a Gradio interface with a dropdown.

```python
def plot_platform(platform):
    # Step 1: Filter the data
    filtered = ___  # Filter music where platform matches
    
    # Step 2: Create the chart
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(filtered["student"], filtered["hours_per_week"], color="coral")
    ax.set_title(f"Listening Hours — {platform} Users")
    ax.set_xlabel("Student")
    ax.set_ylabel("Hours per Week")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    
    # Step 3: Return the figure
    return ___

# Create and launch the interface
platforms = music["platform"].unique().tolist()

demo = gr.Interface(
    fn=___,
    inputs=gr.Dropdown(choices=___, value=platforms[0], label="Choose a platform"),
    outputs=gr.Plot(label="Listening Hours")
)

demo.launch()
```

<details>
<summary>💡 Click for solution</summary>

```python
def plot_platform(platform):
    filtered = music[music["platform"] == platform]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(filtered["student"], filtered["hours_per_week"], color="coral")
    ax.set_title(f"Listening Hours — {platform} Users")
    ax.set_xlabel("Student")
    ax.set_ylabel("Hours per Week")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    
    return fig

platforms = music["platform"].unique().tolist()

demo = gr.Interface(
    fn=plot_platform,
    inputs=gr.Dropdown(choices=platforms, value=platforms[0], label="Choose a platform"),
    outputs=gr.Plot(label="Listening Hours")
)

demo.launch()
```
</details>

---

## Part 4: Building the Dashboard 🖥️

Now we'll combine multiple inputs into a single dashboard. Instead of one dropdown, we'll give users:
- A **dropdown** to pick a genre
- A **dropdown** to pick a platform
- A **checkbox** to show only students who would recommend their music

The function will filter the data based on **all three choices** and return both a **summary text** and a **chart**.

### Multiple Inputs = Multiple Parameters

When you add more inputs, each one maps to a **parameter** in your function — just like Lesson 6!

```python
# One input = one parameter
def my_function(genre):        # gr.Dropdown → genre
    ...

# Three inputs = three parameters  
def my_function(genre, platform, recommend_only):  
#                  ↑        ↑           ↑
#              Dropdown  Dropdown    Checkbox
```

### Multiple Outputs

To return more than one thing, your function returns a **tuple** (multiple values separated by commas), and you give Gradio a **list** of output types:

```python
def my_function(...):
    text = "some summary"
    fig = ...  # a chart
    return text, fig  # Return both!

demo = gr.Interface(
    fn=my_function,
    inputs=[...],
    outputs=[gr.Textbox(...), gr.Plot(...)]  # List of outputs
)
```

### The Dashboard Function

Let's put it all together:

```python
def music_dashboard(genre, platform, recommend_only):
    # Start with the full dataset
    filtered = music.copy()
    
    # Apply genre filter (if not "All")
    if genre != "All":
        filtered = filtered[filtered["genre"] == genre]
    
    # Apply platform filter (if not "All")
    if platform != "All":
        filtered = filtered[filtered["platform"] == platform]
    
    # Apply recommendation filter (if checkbox is ticked)
    if recommend_only:
        filtered = filtered[filtered["would_recommend"] == "Yes"]
    
    # Build the summary text
    count = len(filtered)
    if count == 0:
        return "No students match these filters. Try a different combination!", None
    
    avg_hours = filtered["hours_per_week"].mean()
    top_listener = filtered.sort_values("hours_per_week", ascending=False).iloc[0]
    
    summary = f"📊 Results: {count} students found\n"
    summary += f"📈 Average listening: {avg_hours:.1f} hours/week\n"
    summary += f"🏆 Top listener: {top_listener['student']} ({top_listener['hours_per_week']} hrs/week)\n"
    summary += f"🎤 Favourite artist of top listener: {top_listener['favourite_artist']}"
    
    # Build the chart
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Colour bars by mood
    mood_colors = {"Energetic": "#FF6B6B", "Chill": "#4ECDC4", "Mixed": "#FFD93D"}
    colors = [mood_colors.get(m, "steelblue") for m in filtered["mood"]]
    
    ax.bar(filtered["student"], filtered["hours_per_week"], color=colors)
    ax.set_title("Listening Hours by Student (coloured by mood)")
    ax.set_xlabel("Student")
    ax.set_ylabel("Hours per Week")
    ax.axhline(y=avg_hours, color="black", linestyle="--", alpha=0.5, label=f"Average ({avg_hours:.1f})")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    
    return summary, fig

# Build the input options
genre_choices = ["All"] + music["genre"].unique().tolist()
platform_choices = ["All"] + music["platform"].unique().tolist()

demo = gr.Interface(
    fn=music_dashboard,
    inputs=[
        gr.Dropdown(choices=genre_choices, value="All", label="🎵 Genre"),
        gr.Dropdown(choices=platform_choices, value="All", label="📱 Platform"),
        gr.Checkbox(label="Show only students who recommend their music")
    ],
    outputs=[
        gr.Textbox(label="Summary"),
        gr.Plot(label="Chart")
    ],
    title="🎵 Music Streaming Dashboard",
    description="Explore our class music survey! Filter by genre, platform, and recommendations."
)

demo.launch()
```

🎉 **That's a full interactive dashboard!** Try different combinations:
- Select "Hip-Hop" + "All" platforms — who listens the most?
- Select "All" genres + "Spotify" — how many Spotify users are there?
- Tick the recommendation checkbox — does it change the average?

### What's Happening Under the Hood

Every time you change a dropdown or tick the checkbox, Gradio:
1. Reads the values from all three inputs
2. Calls `music_dashboard(genre, platform, recommend_only)` with those values
3. Displays the returned summary text and chart

The function is doing the same pandas filtering you learned in Lesson 7 — the only difference is that **the user controls the filter values** instead of you hardcoding them.

---

### Gradio Input Types Reference

Here are the most useful Gradio components. Each one sends a different type of data to your function:

![Gradio Components Visual Reference](https://raw.githubusercontent.com/simunaqv2/python-jupyter-course/main/lessons/lesson09/gradio_components.png)

**Input components** (what the user interacts with):

| Component | What it looks like | Data it sends to your function |
|-----------|-------------------|-------------------------------|
| `gr.Textbox()` | Text input box | A string |
| `gr.Slider(minimum, maximum, step)` | Draggable slider | A number (int or float) |
| `gr.Dropdown(choices=[...])` | Dropdown menu | A string (the selected option) |
| `gr.Checkbox()` | Tick box | `True` or `False` |
| `gr.Radio(choices=[...])` | Radio buttons | A string (the selected option) |

**Output components** (what Gradio displays):

| Component | What it displays |
|-----------|------------------|
| `gr.Textbox()` | Text |
| `gr.Plot()` | A matplotlib figure |
| `gr.Dataframe()` | A pandas DataFrame as a table |

---

## Part 5: Returning DataFrames 📋

Gradio can also display DataFrames directly as interactive tables. This is great for showing the raw data behind a chart.

Let's build a quick student lookup tool:

```python
def lookup_genre(genre):
    if genre == "All":
        result = music[["student", "favourite_artist", "genre", "hours_per_week", "mood"]]
    else:
        filtered = music[music["genre"] == genre]
        result = filtered[["student", "favourite_artist", "hours_per_week", "mood"]]
    
    return result

genre_choices = ["All"] + music["genre"].unique().tolist()

demo = gr.Interface(
    fn=lookup_genre,
    inputs=gr.Dropdown(choices=genre_choices, value="All", label="Genre"),
    outputs=gr.Dataframe(label="Students")
)

demo.launch()
```

Notice that `gr.Dataframe()` takes a pandas DataFrame directly — no conversion needed. Gradio turns it into a nice, scrollable table in the web app.

---

## Part 6: Mini Project — Extend the Dashboard 🏆

Now it's your turn! Choose **one** of the following features to add to the music dashboard. Use the starter code below and fill in the function.

### Option A: Genre Comparison Chart

Build a Gradio app that shows a **grouped bar chart** of average listening hours by genre. Add a slider that lets the user set a minimum hours threshold — only genres with an average above the threshold appear in the chart.

### Option B: Student Profile Finder

Build a Gradio app with a dropdown of student names. When a student is selected, return their full profile as formatted text (name, age, favourite artist, genre, hours, platform, mood, recommendation).

### Option C: Top Listeners Leaderboard

Build a Gradio app with a slider for "Top N" (from 3 to 15). Return a horizontal bar chart showing the top N listeners by hours per week, plus a DataFrame showing their details.

Pick whichever option interests you most!

In [ ]:
# MINI PROJECT: Choose Option A, B, or C and build it here!

# Your function goes here


# Your Gradio interface goes here


<details>
<summary>💡 Click for Option A solution</summary>

```python
def genre_comparison(min_hours):
    genre_avg = music.groupby("genre")["hours_per_week"].mean()
    genre_avg = genre_avg[genre_avg >= min_hours].sort_values(ascending=True)
    
    if len(genre_avg) == 0:
        return "No genres meet this threshold.", None
    
    fig, ax = plt.subplots(figsize=(8, 5))
    genre_avg.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"Average Listening Hours by Genre (min: {min_hours} hrs)")
    ax.set_xlabel("Average Hours per Week")
    ax.set_ylabel("Genre")
    plt.tight_layout()
    
    summary = f"Genres shown: {len(genre_avg)} of {music['genre'].nunique()}"
    return summary, fig

demo = gr.Interface(
    fn=genre_comparison,
    inputs=gr.Slider(minimum=0, maximum=20, step=1, value=5, label="Minimum average hours"),
    outputs=[gr.Textbox(label="Summary"), gr.Plot(label="Genre Comparison")],
    title="🎵 Genre Comparison"
)

demo.launch()
```
</details>

<details>
<summary>💡 Click for Option B solution</summary>

```python
def student_profile(student_name):
    student = music[music["student"] == student_name].iloc[0]
    
    profile = f"🎧 Music Profile: {student['student']}\n"
    profile += f"{'=' * 30}\n"
    profile += f"Age: {student['age']}\n"
    profile += f"Favourite Artist: {student['favourite_artist']}\n"
    profile += f"Genre: {student['genre']}\n"
    profile += f"Listening: {student['hours_per_week']} hours/week\n"
    profile += f"Platform: {student['platform']}\n"
    profile += f"Discovery Method: {student['discover_method']}\n"
    profile += f"Mood: {student['mood']}\n"
    profile += f"Would Recommend: {student['would_recommend']}"
    
    return profile

student_names = sorted(music["student"].tolist())

demo = gr.Interface(
    fn=student_profile,
    inputs=gr.Dropdown(choices=student_names, value=student_names[0], label="Choose a student"),
    outputs=gr.Textbox(label="Profile", lines=10),
    title="🎧 Student Music Profile"
)

demo.launch()
```
</details>

<details>
<summary>💡 Click for Option C solution</summary>

```python
def top_listeners(n):
    n = int(n)
    top = music.sort_values("hours_per_week", ascending=False).head(n)
    
    # Chart (reversed so #1 is at the top)
    top_reversed = top.sort_values("hours_per_week", ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(4, n * 0.4)))
    ax.barh(top_reversed["student"], top_reversed["hours_per_week"], color="steelblue")
    ax.set_title(f"🏆 Top {n} Listeners")
    ax.set_xlabel("Hours per Week")
    plt.tight_layout()
    
    # Table
    table = top[["student", "favourite_artist", "genre", "hours_per_week", "platform"]].reset_index(drop=True)
    table.index = table.index + 1  # Rank from 1
    
    return fig, table

demo = gr.Interface(
    fn=top_listeners,
    inputs=gr.Slider(minimum=3, maximum=15, step=1, value=5, label="How many top listeners?"),
    outputs=[gr.Plot(label="Leaderboard"), gr.Dataframe(label="Details")],
    title="🏆 Listening Leaderboard"
)

demo.launch()
```
</details>

---

## ⚠️ Common Mistakes

| Mistake | Example | Fix |
|---------|---------|-----|
| Forgetting to install Gradio | `ModuleNotFoundError: No module named 'gradio'` | Run `!pip install gradio -q` first |
| Using `plt.show()` instead of returning figure | Chart appears empty in Gradio | Use `fig, ax = plt.subplots()` and `return fig` |
| Function returns nothing | Output shows `None` | Add a `return` statement (remember Lesson 6!) |
| Wrong number of parameters | `TypeError: function takes 1 argument but 3 were given` | Your function needs one parameter per Gradio input |
| Forgetting `plt.tight_layout()` | Labels get cut off in the chart | Add `plt.tight_layout()` before `return fig` |

---

## 📚 Key Vocabulary

| Term | Definition |
|------|------------|
| **Gradio** | A Python library that turns functions into interactive web apps |
| **Interface** | A Gradio object that connects inputs, a function, and outputs |
| **Component** | A UI element like a Textbox, Slider, Dropdown, or Plot |
| **Launch** | Starting the web app so users can interact with it |
| **Figure (`fig`)** | A matplotlib chart object that can be returned to Gradio for display |

---

## 📝 Homework

Complete the following tasks using the `music` DataFrame.

**Remember:** Make sure you've run the cells that install Gradio and load the CSV first!

---

### Task 1: Top Listeners Tab ⭐

**Goal:** Build a Gradio app that shows a sorted horizontal bar chart of the top 5 students with the most listening hours.

**Your app should:**
1. Sort `music` by `hours_per_week` (highest first)
2. Take the top 5 rows
3. Create a horizontal bar chart (remember to reverse for display!)
4. Return the chart in a Gradio interface

**Helpful code snippets:**

Sorting and getting top rows:
```python
top_5 = music.sort_values("hours_per_week", ascending=False).head(5)
```

Creating a horizontal bar chart that returns a figure:
```python
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top_5["student"], top_5["hours_per_week"], color="steelblue")
ax.set_title("Top 5 Listeners")
ax.set_xlabel("Hours per Week")
plt.tight_layout()
return fig
```

A Gradio interface with no inputs (just a button):
```python
demo = gr.Interface(
    fn=your_function,
    inputs=[],
    outputs=gr.Plot(label="Top Listeners")
)
```

In [ ]:
# HOMEWORK Task 1: Top Listeners
import matplotlib.pyplot as plt

# Step 1: Write a function that creates the top 5 chart


# Step 2: Create the Gradio interface


# Step 3: Launch it


<details>
<summary>💡 Click for solution</summary>

```python
import matplotlib.pyplot as plt

def top_5_chart():
    top_5 = music.sort_values("hours_per_week", ascending=False).head(5)
    top_5_reversed = top_5.sort_values("hours_per_week", ascending=True)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top_5_reversed["student"], top_5_reversed["hours_per_week"], color="steelblue")
    ax.set_title("🏆 Top 5 Listeners by Hours per Week")
    ax.set_xlabel("Hours per Week")
    plt.tight_layout()
    
    return fig

demo = gr.Interface(
    fn=top_5_chart,
    inputs=[],
    outputs=gr.Plot(label="Top Listeners")
)

demo.launch()
```
</details>

---

### Task 2: Dual Filter Explorer ⭐⭐

**Goal:** Build a Gradio app where the user picks both a **genre** and a **platform**, and sees the matching students plus their average hours.

**Your app should:**
1. Accept two dropdowns: genre and platform (both with an "All" option)
2. Filter the dataset based on both selections
3. Return a text summary (count and average hours) and a bar chart

**Helpful code snippets:**

Applying multiple filters:
```python
filtered = music.copy()
if genre != "All":
    filtered = filtered[filtered["genre"] == genre]
if platform != "All":
    filtered = filtered[filtered["platform"] == platform]
```

Building dropdown choices with "All":
```python
genre_choices = ["All"] + music["genre"].unique().tolist()
```

In [ ]:
# HOMEWORK Task 2: Dual Filter Explorer
import matplotlib.pyplot as plt

# Step 1: Write the function (takes genre and platform, returns text + chart)


# Step 2: Set up the dropdown choices


# Step 3: Create and launch the Gradio interface


<details>
<summary>💡 Click for solution</summary>

```python
import matplotlib.pyplot as plt

def dual_filter(genre, platform):
    filtered = music.copy()
    
    if genre != "All":
        filtered = filtered[filtered["genre"] == genre]
    if platform != "All":
        filtered = filtered[filtered["platform"] == platform]
    
    count = len(filtered)
    if count == 0:
        return "No students match these filters.", None
    
    avg = filtered["hours_per_week"].mean()
    summary = f"Found {count} students\nAverage listening: {avg:.1f} hours/week"
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(filtered["student"], filtered["hours_per_week"], color="coral")
    ax.set_title(f"Listening Hours — Genre: {genre}, Platform: {platform}")
    ax.set_xlabel("Student")
    ax.set_ylabel("Hours per Week")
    ax.axhline(y=avg, color="black", linestyle="--", alpha=0.5, label=f"Average ({avg:.1f})")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    
    return summary, fig

genre_choices = ["All"] + music["genre"].unique().tolist()
platform_choices = ["All"] + music["platform"].unique().tolist()

demo = gr.Interface(
    fn=dual_filter,
    inputs=[
        gr.Dropdown(choices=genre_choices, value="All", label="Genre"),
        gr.Dropdown(choices=platform_choices, value="All", label="Platform")
    ],
    outputs=[
        gr.Textbox(label="Summary"),
        gr.Plot(label="Chart")
    ],
    title="🎵 Dual Filter Explorer"
)

demo.launch()
```
</details>

---

### 🌟 Stretch Challenge: Music Profile Finder ⭐⭐⭐

**Goal:** Build an app where the user types a student name and gets a profile card showing all their data, plus their **percentile ranking** for listening hours.

**Your app should:**
1. Accept a student name from a dropdown
2. Look up that student's full row in the DataFrame
3. Calculate what percentage of students they listen more than
4. Return a formatted profile card as text

**Hints:**

Getting a single student's row:
```python
student = music[music["student"] == name].iloc[0]
```

Calculating percentile (what % of students listen less):
```python
all_hours = music["hours_per_week"]
below = len(all_hours[all_hours < student_hours])
percentile = (below / len(all_hours)) * 100
```

In [ ]:
# STRETCH CHALLENGE: Music Profile Finder

# Your function here


# Your Gradio interface here


<details>
<summary>💡 Click for solution</summary>

```python
def music_profile(name):
    student = music[music["student"] == name].iloc[0]
    
    # Calculate percentile
    all_hours = music["hours_per_week"]
    below = len(all_hours[all_hours < student["hours_per_week"]])
    percentile = (below / len(all_hours)) * 100
    
    profile = f"🎧 Music Profile: {student['student']}\n"
    profile += f"{'=' * 35}\n"
    profile += f"  Age:              {student['age']}\n"
    profile += f"  Favourite Artist:  {student['favourite_artist']}\n"
    profile += f"  Genre:            {student['genre']}\n"
    profile += f"  Hours/Week:       {student['hours_per_week']}\n"
    profile += f"  Platform:         {student['platform']}\n"
    profile += f"  Discovers Music:  {student['discover_method']}\n"
    profile += f"  Mood:             {student['mood']}\n"
    profile += f"  Would Recommend:  {student['would_recommend']}\n"
    profile += f"{'=' * 35}\n"
    profile += f"  📊 Listens more than {percentile:.0f}% of students"
    
    return profile

student_names = sorted(music["student"].tolist())

demo = gr.Interface(
    fn=music_profile,
    inputs=gr.Dropdown(choices=student_names, value=student_names[0], label="Choose a student"),
    outputs=gr.Textbox(label="Profile", lines=12),
    title="🎧 Music Profile Finder"
)

demo.launch()
```
</details>

---

## 🎯 What's Next?

You can now build interactive web apps that let anyone explore data — no Python knowledge required! This is a skill that data scientists use daily to share their work.

**Next lesson:** We'll start planning your **capstone project** — a chance to pick a topic you care about, find or create a dataset, and build your own interactive dashboard from scratch. This will be your portfolio piece to show off everything you've learned! 🚀